In [ ]:
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv
load_dotenv(r"C:\Users\Administrator\Documents\1st Project\Streamlit\.env")

username = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST", "localhost")
port = os.getenv("DB_PORT", "3306")
database = os.getenv("DB_NAME", "cart2insights")

engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}")

In [ ]:
import pandas as pd

processed_path = r"C:\Users\Administrator\Documents\1st Project\Processed_data"

tables = {
    "customers": "customers_clean.csv",
    "geolocation": "geolocation_clean.csv",
    "order_items": "order_items_clean.csv",
    "order_payments": "order_payments_clean.csv",
    "order_reviews": "order_reviews_clean.csv",
    "orders": "orders_clean.csv",
    "products": "products_clean.csv",
    "sellers": "sellers_clean.csv",
    "category_translation": "category_translation_clean.csv",
}

for table_name, filename in tables.items():
    df = pd.read_csv(f"{processed_path}\\{filename}")
    df.to_sql(table_name, con=engine, if_exists="replace", index=False)
    print(f"Loaded {table_name}: {df.shape[0]} rows")

In [ ]:
mismatch = pd.read_sql("""
    SELECT DISTINCT p.product_category_name
    FROM products p
    LEFT JOIN category_translation c
    ON p.product_category_name = c.product_category_name
    WHERE c.product_category_name IS NULL;
""", con=engine)
print(mismatch)

In [ ]:
with engine.connect() as conn:
    conn.execute(text("""
        INSERT INTO category_translation (product_category_name, product_category_name_english)
        VALUES 
            ('unknown', 'unknown'),
            ('pc_gamer', 'pc_gamer'),
            ('portateis_cozinha_e_preparadores_de_alimentos', 'portable_kitchen_and_food_preparation')
    """))
    conn.commit()

In [ ]:
check = pd.read_sql("SELECT product_category_name, COUNT(*) as cnt FROM category_translation GROUP BY product_category_name HAVING cnt > 1;", con=engine)
print(check)

In [ ]:
pk_statements = [
    "ALTER TABLE customers MODIFY customer_id VARCHAR(100);",
    "ALTER TABLE customers ADD PRIMARY KEY (customer_id);",
    "ALTER TABLE orders MODIFY order_id VARCHAR(100);",
    "ALTER TABLE orders ADD PRIMARY KEY (order_id);",
    "ALTER TABLE products MODIFY product_id VARCHAR(100);",
    "ALTER TABLE products ADD PRIMARY KEY (product_id);",
    "ALTER TABLE sellers MODIFY seller_id VARCHAR(100);",
    "ALTER TABLE sellers ADD PRIMARY KEY (seller_id);",
    "ALTER TABLE category_translation MODIFY product_category_name VARCHAR(100);",
    "ALTER TABLE category_translation ADD PRIMARY KEY (product_category_name);",
    "ALTER TABLE order_reviews ADD PRIMARY KEY (review_pk);",
    "ALTER TABLE order_items MODIFY order_id VARCHAR(100);",
    "ALTER TABLE order_items ADD PRIMARY KEY (order_id, order_item_id);",
    "ALTER TABLE order_payments MODIFY order_id VARCHAR(100);",
    "ALTER TABLE order_payments ADD PRIMARY KEY (order_id, payment_sequential);",
]

with engine.connect() as conn:
    for stmt in pk_statements:
        conn.execute(text(stmt))
        print("Done:", stmt)
    conn.commit()

In [ ]:
fk_statements = [
    "ALTER TABLE orders MODIFY customer_id VARCHAR(100);",
    "ALTER TABLE orders ADD FOREIGN KEY (customer_id) REFERENCES customers(customer_id);",
    "ALTER TABLE order_items MODIFY product_id VARCHAR(100);",
    "ALTER TABLE order_items ADD FOREIGN KEY (order_id) REFERENCES orders(order_id);",
    "ALTER TABLE order_items ADD FOREIGN KEY (product_id) REFERENCES products(product_id);",
    "ALTER TABLE order_items MODIFY seller_id VARCHAR(100);",
    "ALTER TABLE order_items ADD FOREIGN KEY (seller_id) REFERENCES sellers(seller_id);",
    "ALTER TABLE order_payments ADD FOREIGN KEY (order_id) REFERENCES orders(order_id);",
    "ALTER TABLE order_reviews MODIFY order_id VARCHAR(100);",
    "ALTER TABLE order_reviews ADD FOREIGN KEY (order_id) REFERENCES orders(order_id);",
    "ALTER TABLE products MODIFY product_category_name VARCHAR(100);",
    "ALTER TABLE products ADD FOREIGN KEY (product_category_name) REFERENCES category_translation(product_category_name);",
]

with engine.connect() as conn:
    for stmt in fk_statements:
        conn.execute(text(stmt))
        print("Done:", stmt)
    conn.commit()

In [ ]:
for table in ["customers", "geolocation", "orders", "order_items", "order_payments",
              "order_reviews", "products", "sellers", "category_translation"]:
    result = pd.read_sql(f"SHOW CREATE TABLE {table};", con=engine)
    print(f"===== {table} =====")
    print(result['Create Table'][0])
    print()